# Fashion Brand Collaboration Recommendation System

## Objective
Build a recommendation system that suggests the most suitable fashion brands for collaboration based on multiple business features using Cosine Similarity.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_excel("../data/clean_fashion_brands.xlsx")

In [3]:
df.head()

,Brand,Category,Country,Followers_Million,Engagement_Rate,Avg_Price_USD,Sustainability,Target_Audience,Global_Stores,Trend_Score,Collaboration_Count,Collaboration_Score
0,Nike,Sportswear,USA,27,1.90,166,High,Young Adults,3549,83,9,NaN
1,Adidas,Sportswear,Germany,60,2.63,98,Medium,Adults,4118,86,19,NaN
2,Puma,Sportswear,Germany,79,2.86,88,Medium,Young Adults,1217,75,13,NaN
3,Zara,Fast Fashion,Spain,22,1.35,46,High,Adults,681,84,12,NaN
4,H&M,Fast Fashion,Sweden,15,1.72,65,High,Adults,2386,92,6,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Brand                101 non-null    object 
 1   Category             101 non-null    object 
 2   Country              101 non-null    object 
 3   Followers_Million    101 non-null    int64  
 4   Engagement_Rate      101 non-null    float64
 5   Avg_Price_USD        101 non-null    int64  
 6   Sustainability       101 non-null    object 
 7   Target_Audience      101 non-null    object 
 8   Global_Stores        101 non-null    int64  
 9   Trend_Score          101 non-null    int64  
 10  Collaboration_Count  101 non-null    int64  
 11  Collaboration_Score  0 non-null      float64
dtypes: float64(2), int64(5), object(5)
memory usage: 9.6+ KB


In [5]:
df.shape

(101, 12)

In [6]:
sustainability_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

df["Sustainability_Score"] = df["Sustainability"].map(sustainability_map)

In [7]:
df[["Sustainability","Sustainability_Score"]].head()

,Sustainability,Sustainability_Score
0,High,3
1,Medium,2
2,Medium,2
3,High,3
4,High,3


In [8]:
category_encoder = LabelEncoder()

df["Category_Code"] = category_encoder.fit_transform(df["Category"])

In [9]:
df[["Category","Category_Code"]].head()

,Category,Category_Code
0,Sportswear,8
1,Sportswear,8
2,Sportswear,8
3,Fast Fashion,2
4,Fast Fashion,2


In [10]:
audience_encoder = LabelEncoder()

df["Audience_Code"] = audience_encoder.fit_transform(df["Target_Audience"])

In [11]:
df[["Target_Audience","Audience_Code"]].head()

,Target_Audience,Audience_Code
0,Young Adults,2
1,Adults,0
2,Young Adults,2
3,Adults,0
4,Adults,0


In [12]:
features = [
    "Followers_Million",
    "Engagement_Rate",
    "Avg_Price_USD",
    "Global_Stores",
    "Trend_Score",
    "Collaboration_Count",
    "Sustainability_Score"
]

In [13]:
scaler = MinMaxScaler()

scaled_features = scaler.fit_transform(df[features])

In [14]:
scaled_df = pd.DataFrame(scaled_features, columns=features)

scaled_df.head()

,Followers_Million,Engagement_Rate,Avg_Price_USD,Global_Stores,Trend_Score,Collaboration_Count,Sustainability_Score
0,0.076923,0.391061,0.070041,0.726677,0.464286,0.272727,1.0
1,0.182692,0.798883,0.035276,0.847049,0.571429,0.727273,0.5
2,0.243590,0.927374,0.030164,0.233340,0.178571,0.454545,0.5
3,0.060897,0.083799,0.008691,0.119949,0.500000,0.409091,1.0
4,0.038462,0.290503,0.018405,0.480643,0.785714,0.136364,1.0


In [15]:
scaled_df["Collaboration_Score"] = (
      scaled_df["Followers_Million"] * 0.25
    + scaled_df["Engagement_Rate"] * 0.20
    + scaled_df["Trend_Score"] * 0.25
    + scaled_df["Sustainability_Score"] * 0.10
    + scaled_df["Collaboration_Count"] * 0.10
    + scaled_df["Global_Stores"] * 0.10
)

scaled_df["Collaboration_Score"] = (
    scaled_df["Collaboration_Score"] * 100
).round(2)

In [16]:
df["Collaboration_Score"] = scaled_df["Collaboration_Score"]

In [17]:
df[["Brand", "Collaboration_Score"]].head()

,Brand,Collaboration_Score
0,Nike,41.35
1,Adidas,55.57
2,Puma,40.98
3,Zara,30.99
4,H&M,42.58


In [18]:
similarity_matrix = cosine_similarity(scaled_df[features])

brand_index = pd.Series(df.index, index=df["Brand"])

In [19]:
similarity_matrix.shape

(101, 101)

In [26]:
def recommend_brand(brand_name, top_n=5):

    if brand_name not in brand_index.index:
        return "Brand not found!"

    idx = brand_index[brand_name]

    similarity_scores = list(enumerate(similarity_matrix[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    recommendations = []

    for i, score in similarity_scores:

        reason = []

        if df.iloc[i]["Trend_Score"] >= 80:
            reason.append("High Trend")

        if df.iloc[i]["Engagement_Rate"] >= 2:
            reason.append("Strong Social Presence")

        if df.iloc[i]["Sustainability"] == "High":
            reason.append("Highly Sustainable")

        if df.iloc[i]["Collaboration_Score"] >= 45:
            reason.append("High Collaboration Potential")

        if df.iloc[i]["Global_Stores"] >= 1000:
            reason.append("Strong Global Presence")

        if len(reason) == 0:
            reason.append("Good Business Match")

        recommendations.append({

            "Brand": df.iloc[i]["Brand"],

            "Similarity Score": round(score,3),

            "Category": df.iloc[i]["Category"],

            "Trend Score": df.iloc[i]["Trend_Score"],

            "Sustainability": df.iloc[i]["Sustainability"],

            "Collaboration Score": df.iloc[i]["Collaboration_Score"],

            "Recommendation Reason": ", ".join(reason)

        })

    recommendation_df = pd.DataFrame(recommendations)

    return recommendation_df

In [27]:
recommend_brand("Nike")

,Brand,Similarity Score,Category,Trend Score,Sustainability,Collaboration Score,Recommendation Reason
0,Birkenstock,0.965,Footwear,76,Medium,25.22,Strong Global Presence
1,Tory Burch,0.958,Premium,84,High,35.40,"High Trend, Highly Sustainable, Strong Global ..."
2,Celio,0.951,Casual,86,High,48.70,"High Trend, Highly Sustainable, High Collabora..."
3,H&M,0.950,Fast Fashion,92,High,42.58,"High Trend, Highly Sustainable, Strong Global ..."
4,BAPE,0.945,Streetwear,84,High,49.14,"High Trend, Strong Social Presence, Highly Sus..."


In [28]:
recommend_brand("Gucci")

,Brand,Similarity Score,Category,Trend Score,Sustainability,Collaboration Score,Recommendation Reason
0,Louis Vuitton,0.931,Luxury,76,Medium,27.49,Strong Global Presence
1,Celine,0.900,Luxury,77,Medium,31.07,Strong Global Presence
2,Kenzo,0.879,Luxury,93,High,52.87,"High Trend, Strong Social Presence, Highly Sus..."
3,Versace,0.878,Luxury,93,Medium,48.98,"High Trend, High Collaboration Potential, Stro..."
4,Chanel,0.876,Luxury,71,Medium,26.45,Strong Global Presence


In [29]:
recommend_brand("Zara")

,Brand,Similarity Score,Category,Trend Score,Sustainability,Collaboration Score,Recommendation Reason
0,Tory Burch,0.955,Premium,84,High,35.40,"High Trend, Highly Sustainable, Strong Global ..."
1,Desigual,0.943,Casual,74,High,23.68,Highly Sustainable
2,Shein,0.940,Fast Fashion,89,High,44.87,"High Trend, Highly Sustainable"
3,Van Heusen,0.938,Formal,81,High,39.75,"High Trend, Strong Social Presence, Highly Sus..."
4,Jack & Jones,0.934,Casual,80,High,31.05,"High Trend, Highly Sustainable, Strong Global ..."


In [30]:
recommend_brand("Chanel")

,Brand,Similarity Score,Category,Trend Score,Sustainability,Collaboration Score,Recommendation Reason
0,Prada,0.967,Luxury,72,Medium,24.54,Good Business Match
1,Fendi,0.961,Luxury,72,High,39.99,"Highly Sustainable, Strong Global Presence"
2,Dior,0.948,Luxury,80,Medium,47.18,"High Trend, Strong Social Presence, High Colla..."
3,Louis Vuitton,0.898,Luxury,76,Medium,27.49,Strong Global Presence
4,Celine,0.883,Luxury,77,Medium,31.07,Strong Global Presence


In [31]:
recommend_brand("ABC")

'Brand not found!'

In [32]:
df.to_excel("../data/final_brand_recommendation_dataset.xlsx", index=False)

print("Dataset Saved Successfully!")

Dataset Saved Successfully!
